# DB증권 Open API — 해외주식(미국) 트레이딩 기초 (ov-stock-trading-basic)

`dbsec_sdk`(비동기 클라이언트)로 **미국주식 매매의 기본 흐름을 처음부터 끝까지** 실습하는 노트북입니다 —
시세·차트 조회부터 계좌(잔고/증거금·예수금) 확인, 주문가능금액, 주문(시장가/지정가), 체결 조회,
정정·취소, 동시 조회, 실시간 수신까지 한 번에 따라 할 수 있습니다.

**다루는 내용**

| # | 섹션 | API Endpoint | TR |
|---|---|---|---|
| 1 | 셋업 — 비동기 클라이언트 생성 | - | - |
| 2 | 접근토큰 (발급/캐시 · auto_token) | `/oauth2/token` | token |
| 3 | 시세 — 현재가 / 호가 | `/api/v1/quote/overseas-stock/inquiry/price`<br>`/api/v1/quote/overseas-stock/inquiry/orderbook` | FSTKPRICE<br>FSTKHOGA |
| 4 | 차트 — 일차트 | `/api/v1/quote/overseas-stock/chart/day` | FSTKCHARTDAY |
| 5 | 계좌 — 잔고/증거금 / 예수금상세 | `/api/v1/trading/overseas-stock/inquiry/balance-margin`<br>`/api/v1/trading/overseas-stock/inquiry/deposit-detail` | CAZCQ00400<br>CAZCQ01400 |
| 6 | 주문가능금액 | `/api/v1/trading/overseas-stock/inquiry/able-orderqty` | CAZCQ01300 |
| 7 | 주문 — 시장가 / 지정가 매수 | `/api/v1/trading/overseas-stock/order` | CAZCT00100 |
| 8 | 체결/미체결 조회 | `/api/v1/trading/overseas-stock/inquiry/transaction-history` | CAZCQ00100 |
| 9 | 정정 / 취소 (주문 API 통합) | `/api/v1/trading/overseas-stock/order` | CAZCT00100 |
| 10 | 동시 조회 (`asyncio.gather`) | - | - |
| 11 | 실시간 WebSocket (참고) | `wss://openapi.dbsec.co.kr:7070/websocket` | V10 / V60 |

> ⚠️ **주문(7·9번) 셀은 실제 매매가 실행됩니다.** 기본값은 `RUN_ORDER_EXAMPLES = False` 라서
> 그냥 전체 실행(Run All)해도 주문은 **스킵**됩니다. 주문까지 실습하려면 반드시
> **모의투자(`config.yaml` 의 `mode: "demo"`)** 로 바꾼 뒤 플래그를 `True` 로 변경하세요.

**해외주식만의 사전 준비**

- **해외주식 거래신청** 필요 — HTS `[7322] 해외주식 시작하기` / MTS `해외주식 > 서비스신청 > 해외주식 거래신청, 통합증거금 이용신청, 해외ETP 거래신청`
- 시세는 기본 **15분 지연** — 실시간 시세는 MTS/HTS 에서 **실시간 시세 신청(무료)** 후 이용 가능
- **미국 거래시간(한국시간)** — 정규장 23:30~익일 06:00 (서머타임 22:30~05:00) · 프리장 18:00~23:30 (17:00~22:30) · 애프터장 06:00~07:00 (05:00~07:00)

```bash
pip install -e .            # dbsec_sdk 설치
pip install pandas jupyter
cp config.yaml.example config.yaml   # 앱키 입력 + mode: demo
```

## 1. 셋업 — 비동기 클라이언트 생성

`DBSecClient` 는 **비동기** 클라이언트입니다. 노트북(IPython)은 셀에서 **top-level `await`** 를
지원하므로 `asyncio.run()` 없이 바로 `await` 하면 됩니다.

클라이언트가 처리해 주는 것:
- **유량제어(선제)**: 앱 20TPS + 엔드포인트별 TPS 2-tier 자동 페이싱 (`IGW00201` 사전 차단)
- **유량제어(반응형)**: 그래도 `IGW00201` 이 오면 **지수백오프(1·2·4·8초) 재시도**로 회복
- **토큰 자동발급(`auto_token`, 기본 True)**: 토큰이 없거나 무효/만료(`IGW00121`/`00123`)면 자동 발급·재발급(stdin 프롬프트 없음)

> 아래 셀에서 유량제어·토큰 옵션을 **토글**로 노출했습니다 — `rate_limit_safety`(선제 페이싱 강도, 기본 `0.9`),
> `rate_limit_backoff`(IGW00201 지수백오프 재시도, 기본 `True`), `auto_token`(토큰 자동 발급/재발급, 기본 `True`).
> 운영 권장은 **0.9 + 백오프 ON** (실측: safety 1.0=20TPS지만 호출초과↑, 0.9=18TPS·여유, 0.8=16TPS·보수적).

In [1]:
import asyncio, sys, pathlib

# 레포 루트를 import 경로에 추가 (노트북이 dbsec_sdk/client_examples/ 안에 있으므로 위로 올라가며 탐색)
ROOT = pathlib.Path.cwd()
while not (ROOT / "dbsec_sdk").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "dbsec_sdk").exists(), "레포 루트를 찾지 못했습니다 — 노트북을 dbsec_sdk/client_examples/ 폴더에서 여세요."
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.unicode.east_asian_width", True)   # 한글 컬럼 정렬

from dbsec_sdk import DBSecClient

client = DBSecClient(
    str(ROOT / "config.yaml"),
    # ── 유량제어 / 토큰 옵션 (기본값을 명시적으로 노출 — 필요 시 토글) ──
    rate_limit=True,          # 선제 페이싱: 호출 전 TPS 간격을 맞춰 IGW00201(호출초과)을 예방. False면 페이싱 끔.
    rate_limit_safety=0.9,    # 안전계수(0<s≤1): 실제율 = 서버TPS × s.  0.9=90%(10% 여유, 권장) /
                              #   1.0=마진0(처리율 최대지만 순간버스트로 IGW00201↑) / 0.8=보수적(안전, 처리율↓)
    rate_limit_backoff=True,  # 지수백오프 재시도: IGW00201 받으면 1·2·4·8초 후 재시도(선제 페이싱이 놓친 순간버스트를 사후 회복).
                              #   False면 IGW00201을 즉시 APIError로 표면화.
    auto_token=True,          # 토큰 자동 발급/재발급: True면 요청 시 토큰이 없거나 만료/무효(IGW00121/123)면 자동 발급·재발급(프롬프트 없음).
                              #   False면 자동 발급 안 함 → 토큰 없으면 AuthError. get_token()/force_refresh()/revoke()로 직접 관리.
)

SYMBOL = "TSLA"   # 실습 종목 (미국주식/ETF 심볼)
MARKET = "FN"     # FY:뉴욕 FN:나스닥 FA:아멕스
print("mode =", client.config.mode, f" / 실습 종목 = {SYMBOL} ({MARKET})")
print(f"유량제어: rate_limit={client._core._rate.enabled} safety={client._core._rate.safety} backoff={client._core._rate_limit_backoff} | auto_token={client._core._auto_token}")

mode = production  / 실습 종목 = TSLA (FN)
유량제어: rate_limit=True safety=0.9 backoff=True | auto_token=True


In [ ]:
RUN_ORDER_EXAMPLES = False   # ⚠️ True 로 바꾸면 7·9번 주문 셀이 "실제로" 실행됩니다 (모의투자 demo 권장!)

def order_guard() -> bool:
    """주문 셀 공통 가드 — 기본은 스킵, production 모드면 강한 경고."""
    if not RUN_ORDER_EXAMPLES:
        print("⏭ 주문 예제 스킵 — 실행하려면 위 셀의 RUN_ORDER_EXAMPLES = True 로 변경하세요.")
        return False
    if client.config.mode != "demo":
        print("⚠️ 경고: production(실전) 모드입니다 — 아래 주문은 실제 매매로 체결될 수 있습니다!")
    return True

## 2. 접근토큰 — 발급 / 캐시 (auto_token)

토큰은 24시간 유효하며 저장소 루트의 `.dbsec_token.json` 에 캐시됩니다(`examples/` 스크립트와 공유).
유효한 캐시가 있으면 그대로 사용하고, 없으면 `auto_token=True`(기본) 시 자동 발급합니다(stdin 프롬프트 없음).
토큰을 직접 다루려면 `await client.get_token()` / `force_refresh()` / `revoke()` 를 쓰세요.

In [ ]:
token = await client.get_token()
print("토큰 확보 — 앞 16자:", token[:16] + "...  (루트 .dbsec_token.json 에 캐시)")

## 3. 시세 — 현재가 / 호가

- `InputCondMrktDivCode`: `FY` 뉴욕 · `FN` 나스닥 · `FA` 아멕스
- 시세는 기본 **15분 지연** (실시간은 MTS/HTS 에서 실시간 시세 신청 후)
- 미국 가격은 소수점(센트)이 있으므로 `float` 로 다룹니다

In [2]:
# 해외주식현재가조회 (FSTKPRICE, 2 TPS)
resp = await client.apis.ov_stock_quote.ov_stock_inquire_price(
    InputCondMrktDivCode=MARKET, InputIscd1=SYMBOL)
last_price = float(resp.body["Out"]["Prpr"])   # 현재가(USD) — 뒤의 주문가능/지정가 계산에 사용
print(f"status: {resp.status_code} / {SYMBOL} 현재가: ${last_price:,.2f}")
resp.to_dataframe().T.head(12)

status: 200 / TSLA 현재가: $368.37


,0
Sdpr,375.1200
Prpr,368.3700
Mxpr,0.0000
Llam,0.0000
Oprc,374.5100
SdprVrssMrktRate,-0.16
PrprVrssOprcRate,
Hprc,375.0000
SdprVrssHgprRate,-0.03
PrprVrssHgprRate,


In [3]:
# 해외주식호가조회 (FSTKHOGA, 2 TPS) — 매도/매수 5단 호가
resp = await client.apis.ov_stock_quote.ov_stock_inquire_orderbook(
    InputCondMrktDivCode=MARKET, InputIscd1=SYMBOL)
resp.to_dataframe().T.head(15)

,0
Askp1,368.4400
Askp2,368.5000
Askp3,368.6000
Askp4,368.6200
Askp5,368.6500
Bidp1,368.2000
Bidp2,368.1300
Bidp3,368.1200
Bidp4,368.1100
Bidp5,368.1000


## 4. 차트 — 일차트 (최근 1개월)

분/틱차트는 `ov_stock_chart_min` / `ov_stock_chart_tick`, 주/월차트는
`ov_stock_chart_week` / `ov_stock_chart_month` 를 사용하세요.

In [4]:
from datetime import datetime, timedelta

today = datetime.now()
resp = await client.apis.ov_stock_quote.ov_stock_chart_day(
    InputCondMrktDivCode=MARKET, InputIscd1=SYMBOL,
    InputOrgAdjPrc="1",                                          # 1: 수정주가 사용
    InputDate1=(today - timedelta(days=30)).strftime("%Y%m%d"),  # 조회 시작일
    InputDate2=today.strftime("%Y%m%d"))                         # 조회 마감일
resp.to_dataframe().head(10)

,Hour,Date,Prpr,Oprc,Hprc,Lprc,AcmlVol
0,,20260626,368.3900,374.5100,375.0000,367.0600,233767
1,,20260625,375.1200,375.2700,379.1170,371.2210,30259987
2,,20260624,375.5300,380.0800,384.5800,373.0500,37081421
3,,20260623,381.6100,392.6100,392.8700,379.0600,50420210
4,,20260622,405.0500,394.8500,414.7500,394.4000,47819486
5,,20260618,400.4900,398.0950,402.5200,384.7000,58384713
6,,20260617,396.3800,401.5300,405.9400,393.7600,43534299
7,,20260616,404.6600,404.1100,412.4200,400.5400,40255473
8,,20260615,411.1500,412.3700,416.0000,407.1000,45620514
9,,20260612,406.4300,399.4900,406.6800,386.7600,63652286


## 4-B. 연속조회 (자동 페이징) — `fetch_all=True`

같은 해외주식 틱차트(기간 `InputDate1~InputDate2`)를 **① 연속조회 없이(단건)** vs **② `fetch_all=True`** 로 대비합니다.

- **① 단건**: 한 번 호출 = 한 페이지(`dataCnt` 만큼). 데이터가 더 있으면 `has_more=True`/`cont_key` 가 채워지지만 **자동으로 더 가져오지 않습니다**.
- **② `fetch_all=True`**: `cont_yn`/`cont_key` 를 자동 처리하며 끝(`cont_yn='N'`)까지(또는 `max_pages` 까지) 받아 **하나의 `APIResponse` 로 병합**합니다. 원본 페이지는 `resp.pages`.

> 틱차트는 체결이 많아 단건은 `has_more=True`(뒤에 더 있음)가 되고, `fetch_all=True` 는 여러 페이지를 누적합니다.
> 두 셀의 **건수·페이지·`has_more`** 차이를 비교하세요. (틱차트는 방대하니 `fetch_all` 엔 `max_pages` 상한 권장.)

In [ ]:
# ① 연속조회 없이 단건 호출 — 기간을 줘도 한 번 = 한 페이지(dataCnt 만큼)만 받습니다.
#    has_more=True / cont_key 가 있으면 "뒤에 더 있다"는 뜻 (자동으로 안 가져옴 — 아래 ② 와 대비).
from datetime import datetime, timedelta
end = datetime.now() - timedelta(days=1)
while end.weekday() >= 5:                   # 주말 → 직전 평일
    end -= timedelta(days=1)
start = end - timedelta(days=4)             # 약 5일 구간
day1 = start.strftime("%Y%m%d")
day2 = end.strftime("%Y%m%d")

resp = await client.apis.ov_stock_quote.ov_stock_chart_tick(
    InputCondMrktDivCode=MARKET,    # FN:나스닥
    InputIscd1=SYMBOL,              # TSLA
    InputDate1=day1,                # 조회 시작일
    InputDate2=day2,                # 조회 마감일 — 기간지정
    InputHourClsCode="0",
    InputDivXtick="1",              # 1틱 단위
    InputPwDataIncuYn="Y",          # 기간지정 사용
    dataCnt="100",                  # 한 번에 100건
    InputOrgAdjPrc="1",             # 수정주가 사용
)
n_rows = lambda body: next((len(v) for v in body.values() if isinstance(v, list)), 0)
print(f"[단건] {SYMBOL} {day1}~{day2} 틱  →  {n_rows(resp.body):,}건 (1페이지)"
      f"  | has_more={resp.has_more}  cont_key={resp.cont_key[:20]!r}")
display(resp.to_dataframe().head(15))

In [ ]:
# ② fetch_all=True — 연속조회로 cont_yn/cont_key 자동 처리하며 끝까지(또는 max_pages 까지) 받아
#    하나의 APIResponse 로 병합합니다. ①의 단건(1페이지)보다 훨씬 많은 데이터를 한 번에.
from datetime import datetime, timedelta
end = datetime.now() - timedelta(days=1)
while end.weekday() >= 5:                   # 주말 → 직전 평일
    end -= timedelta(days=1)
start = end - timedelta(days=4)
day1 = start.strftime("%Y%m%d")
day2 = end.strftime("%Y%m%d")

resp = await client.apis.ov_stock_quote.ov_stock_chart_tick(
    InputCondMrktDivCode=MARKET,    # FN:나스닥
    InputIscd1=SYMBOL,              # TSLA
    InputDate1=day1,                # 조회 시작일
    InputDate2=day2,                # 조회 마감일 — 기간지정
    InputHourClsCode="0",
    InputDivXtick="1",              # 1틱 단위
    InputPwDataIncuYn="Y",          # 기간지정 사용
    dataCnt="100",                  # 페이지당 100건
    InputOrgAdjPrc="1",             # 수정주가 사용
    fetch_all=True,                 # ← 연속조회 자동 병합 (이게 ①과의 차이)
    max_pages=10,                   # 틱차트는 방대하니 상한 (None/생략 = 끝까지)
)
n_rows = lambda body: next((len(v) for v in body.values() if isinstance(v, list)), 0)
print(f"[fetch_all] {SYMBOL} {day1}~{day2} 틱  →  페이지 {len(resp.pages)}개 / 누적 {n_rows(resp.body):,}건"
      f"  (마지막 has_more={resp.has_more})")
for i, p in enumerate(resp.pages, 1):    # 원본 페이지들은 resp.pages 로 접근
    print(f"  page{i}: {n_rows(p.body):>4,}건   cont_key={p.cont_key[:20]!r}")
display(resp.to_dataframe().head(15))

## 5. 계좌 — 잔고/증거금 / 예수금상세

- 잔고/증거금(CAZCQ00400)의 `TrxTpCode`: `1` 외화잔고 · `2` 주식잔고상세 · `3` 주식잔고(국가별) · `9` 당일실현손익
- 금액은 추정치이며, 원화환산은 가장 최근 최초고시환율 기준

In [ ]:
# 해외주식 잔고/증거금 조회 (CAZCQ00400, 3 TPS) — 주식잔고상세
resp = await client.apis.ov_stock_order.ov_stock_inquire_balance_margin(
    TrxTpCode="2",        # 2: 주식잔고상세
    CmsnTpCode="0",       # 수수료: 0 전부 미포함
    WonFcurrTpCode="2",   # 2: 외화 기준
    DpntBalTpCode="0")    # 소수점잔고: 0 전체
print("status:", resp.status_code, "| 예수금:", resp.body.get("Out", {}).get("Dps"),
      "| 잔고평가:", resp.body.get("Out", {}).get("BalEvalAmt"))
resp.to_dataframe()   # 보유종목 (없으면 빈 DataFrame)

In [ ]:
# 해외주식 예수금상세 (CAZCQ01400, 2 TPS) — 통화별 D+0 ~ D+4 예수금
resp = await client.apis.ov_stock_order.ov_stock_inquire_deposit_detail()
resp.to_dataframe()

## 6. 주문가능금액 — 현재가 기준

In [ ]:
# 해외주식 주문가능금액조회 (CAZCQ01300, 2 TPS)
resp = await client.apis.ov_stock_order.ov_stock_inquire_psbl_amount(
    TrxTpCode="2",                 # 1:매도 2:매수
    AstkIsuNo=SYMBOL,
    AstkOrdPrc=int(last_price),    # 주문가격 (스펙상 정수 — 달러 절사)
    WonFcurrTpCode="2")            # 2: 외화
resp.to_dataframe().T

## 7. 주문 — 시장가 / 지정가 매수 ⚠️

**여기부터 실제 매매 구간입니다.** `RUN_ORDER_EXAMPLES = False`(기본)면 자동 스킵됩니다.

국내와 달리 해외주식은 **주문·정정·취소가 하나의 API**(CAZCT00100, 10 TPS)이며
`OrdTrdTpCode` 로 구분합니다. 주문은 **미국 거래시간(프리/정규/애프터장)에만 접수**됩니다.

| 파라미터 | 의미 | 값 |
|---|---|---|
| `AstkBnsTpCode` | 매매구분 | `1` 매도 / `2` 매수 |
| `AstkOrdprcPtnCode` | 호가유형 | `1` 지정가 / `2` 시장가 / `3` LOO / `4` MOO(매도만) / `5` LOC / `6` MOC(매도만) / `7` VWAP지정가·`8` TWAP지정가(매수/매도) / `9` VWAP시장가·`A` TWAP시장가(매도만) |
| `AstkOrdCndiTpCode` | 주문조건 | `1` FAS(일반) / `2` IOC / `3` FOK |
| `AstkOrdPrc` | 주문가격(USD) | 시장가면 `0` |
| `OrdTrdTpCode` | 주문거래구분 | `0` 주문 / `1` 정정 / `2` 취소 |
| `OrgOrdNo` | 원주문번호 | 신규 주문이면 `0` |

> 시장가 매수는 당사 기준에 따라 현재가 대비 불리한 가격의 지정가로 접수되며,
> 미체결 주문은 애프터장까지 유효합니다.

In [ ]:
# 7-1) 시장가 매수 1주
if order_guard():
    resp = await client.apis.ov_stock_order.ov_stock_order(
        AstkIsuNo=SYMBOL, AstkBnsTpCode="2",
        AstkOrdprcPtnCode="2",     # 2: 시장가
        AstkOrdCndiTpCode="1",     # 1: FAS(일반)
        AstkOrdQty=1, AstkOrdPrc=0,
        OrdTrdTpCode="0", OrgOrdNo=0)
    print(f"[{resp.body.get('rsp_cd')}] 시장가 주문번호:", resp.body["Out"]["OrdNo"])

In [ ]:
# 7-2) 지정가 매수 1주 — 현재가 10% 아래(체결 안 되도록) → 9번 정정/취소 실습용
if order_guard():
    limit_price = int(last_price * 0.90)   # 스펙상 정수(달러 절사)
    resp = await client.apis.ov_stock_order.ov_stock_order(
        AstkIsuNo=SYMBOL, AstkBnsTpCode="2",
        AstkOrdprcPtnCode="1",     # 1: 지정가
        AstkOrdCndiTpCode="1",
        AstkOrdQty=1, AstkOrdPrc=limit_price,
        OrdTrdTpCode="0", OrgOrdNo=0)
    ov_ord_no = int(resp.body["Out"]["OrdNo"])
    print(f"지정가 주문번호: {ov_ord_no} / 주문가격: ${limit_price:,}")

## 8. 체결 / 미체결 조회

In [ ]:
# 해외주식 체결내역조회 (CAZCQ00100, 2 TPS) — 당일 전체
resp = await client.apis.ov_stock_order.ov_stock_inquire_executions(
    QrySrtDt="", QryEndDt="",   # "": 당일 (기간조회는 YYYYMMDD)
    AstkIsuNo="",               # "": 전체 종목
    AstkBnsTpCode="0",          # 0:전체 1:매도 2:매수
    OrdxctTpCode="0",           # 0:전체 1:체결 2:미체결
    StnlnTpCode="0",            # 0:역순
    QryTpCode="1",              # 0:합산별 1:건별
    OnlineYn="0", CvrgOrdYn="0",
    WonFcurrTpCode="2")         # 2:외화
resp.to_dataframe()   # 당일 주문이 없으면 빈 DataFrame

## 9. 정정 / 취소 — 7-2 지정가 주문 대상 ⚠️

국내(별도 API)와 달리 해외주식은 **같은 주문 API** 에 `OrdTrdTpCode="1"`(정정) / `"2"`(취소) 와
`OrgOrdNo`(원주문번호)를 넣어 호출합니다. 정정하면 **새 주문번호**가 발급됩니다.

In [ ]:
# 9-1) 정정 — 지정가를 8% 아래 가격으로 변경
if order_guard():
    if "ov_ord_no" not in globals():
        print("정정할 주문이 없습니다 — 7-2 지정가 주문을 먼저 실행하세요.")
    else:
        new_price = int(last_price * 0.92)
        resp = await client.apis.ov_stock_order.ov_stock_order(
            AstkIsuNo=SYMBOL, AstkBnsTpCode="2",
            AstkOrdprcPtnCode="1", AstkOrdCndiTpCode="1",
            AstkOrdQty=1, AstkOrdPrc=new_price,
            OrdTrdTpCode="1",          # 1: 정정주문
            OrgOrdNo=ov_ord_no)        # 원주문번호
        ov_ord_no = int(resp.body["Out"]["OrdNo"])   # 정정 후 새 주문번호로 갱신
        print(f"정정 완료 — 새 주문번호: {ov_ord_no} / 새 가격: ${new_price:,}")

In [ ]:
# 9-2) 취소
if order_guard():
    if "ov_ord_no" not in globals():
        print("취소할 주문이 없습니다 — 7-2 지정가 주문을 먼저 실행하세요.")
    else:
        resp = await client.apis.ov_stock_order.ov_stock_order(
            AstkIsuNo=SYMBOL, AstkBnsTpCode="2",
            AstkOrdprcPtnCode="1", AstkOrdCndiTpCode="1",
            AstkOrdQty=1, AstkOrdPrc=0,
            OrdTrdTpCode="2",          # 2: 취소주문
            OrgOrdNo=ov_ord_no)
        print("취소 접수:", resp.status_code, "/", resp.message)
        del ov_ord_no

## 10. 보너스 — `asyncio.gather` 동시 조회

서로 다른 엔드포인트는 각자 유량제어 리미터를 쓰므로 **진짜 병렬**로 끝납니다.

In [ ]:
import time, json

t = time.monotonic()
price, balance, deposit = await asyncio.gather(
    client.apis.ov_stock_quote.ov_stock_inquire_price(InputCondMrktDivCode=MARKET, InputIscd1=SYMBOL),
    client.apis.ov_stock_order.ov_stock_inquire_balance_margin(
        TrxTpCode="2", CmsnTpCode="0", WonFcurrTpCode="2", DpntBalTpCode="0"),
    client.apis.ov_stock_order.ov_stock_inquire_deposit_detail(),
)
print(f"3건 동시 조회 {time.monotonic()-t:.2f}s — status:",
      [r.status_code for r in (price, balance, deposit)])

# 조회 결과를 이름별로 줄바꿈해서 출력 (indent=2 → 보기 좋게 / 한 줄로 보려면 indent=None)
results = {"price": price, "balance": balance, "deposit": deposit}
for name, resp in results.items():
    print(f'\n"{name}" :', json.dumps(resp.body, ensure_ascii=False, indent=2))

## 11. 실시간 WebSocket (참고)

해외주식 실시간 TR — `V10` 지연체결가 / `V11` 지연호가 (신청 불필요),
`V60` 체결가 / `V61` 호가 (**실시간 시세 신청 필요**).
`tr_key` 는 **시장코드 + 심볼** (예: 나스닥 테슬라 → `FNTSLA`).

`run()` 이 무한 수신 루프라서 노트북에서는 **커널 Interrupt(■)로 중단**해야 합니다.
연결 문제(세션 수 초과 10011, 1분 6회 연결 제한 등)는 로그(warning)로 원인·해결 힌트가 출력됩니다.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")  # 서버 응답/오류 힌트 표시

if False:   # 실행하려면 True 로 변경 (중단: 커널 Interrupt)
    # 세션이 막혀 있으면(10011) 먼저 초기화: 접속중인 모든 웹소켓 세션을 정리
    await client.apis.ws_common.ws_session_disconnect()

    ws = client.create_websocket()
    ws.on_message(lambda tr_cd, tr_key, data: print(tr_cd, tr_key, data))
    await ws.connect()
    # tr_type 은 직접 지정: "1"=시세구독 "2"=해제 "3"=계좌등록. 일반 시세는 "1".
    await ws.add_realtime(tr_cd="V10", tr_key=MARKET + SYMBOL, tr_type="1")   # 지연체결가 (FNTSLA). 실시간은 V60(신청 필요)
    await ws.run()

---
## 주의사항

> ⚠️ 다시 한 번: 주문 계열 API 는 실제 매매가 실행됩니다. 반드시 **모의투자(demo)** 로 충분히
> 테스트한 뒤 실전(production)으로 전환하세요. 자동매매로 인한 손실 책임은 사용자 본인에게 있습니다.